# Video → 3D (VGGT on Colab)

**Runtime → Change runtime type → T4 GPU** before running.

Important: we clone into `vggt_src` (not `vggt`) so Python does not shadow the real `vggt` package.


In [ ]:
# 1) Install base deps
!pip -q install "numpy<2" Pillow huggingface_hub einops safetensors opencv-python-headless trimesh matplotlib scipy tqdm


In [ ]:
# 2) Clone + install VGGT (folder name must NOT be plain 'vggt')
import os, sys, shutil

# Remove a broken previous clone named 'vggt' if it exists (common cause of this error)
if os.path.isdir("/content/vggt") and not os.path.isdir("/content/vggt/models"):
    print("Removing shadowing /content/vggt folder...")
    shutil.rmtree("/content/vggt")

SRC = "/content/vggt_src"
if not os.path.isdir(SRC):
    !git clone --depth 1 https://github.com/facebookresearch/vggt.git {SRC}

!pip -q install -e {SRC}

# Prefer the installed package path
if SRC in sys.path:
    sys.path.remove(SRC)

import importlib
import vggt
importlib.reload(vggt)
print("vggt loaded from:", vggt.__file__)
from vggt.models.vggt import VGGT
print("VGGT import OK")


In [ ]:
# 3) Upload your video
from google.colab import files
uploaded = files.upload()
assert uploaded, "Upload one video file"
VIDEO_PATH = list(uploaded.keys())[0]
print("Using video:", VIDEO_PATH)


In [ ]:
# 4) Sample frames (short clips work best on free Colab)
import cv2
from pathlib import Path

frames_dir = Path("/content/frames")
frames_dir.mkdir(exist_ok=True)
for p in frames_dir.glob("*"):
    p.unlink()

cap = cv2.VideoCapture(VIDEO_PATH)
fps = cap.get(cv2.CAP_PROP_FPS) or 30
interval = max(1, int(round(fps)))  # ~1 fps
max_frames = 40
idx = saved = 0
while saved < max_frames:
    ok, frame = cap.read()
    if not ok:
        break
    if idx % interval == 0:
        out = frames_dir / f"{saved:06d}.jpg"
        cv2.imwrite(str(out), frame)
        saved += 1
    idx += 1
cap.release()
image_names = sorted(str(p) for p in frames_dir.glob("*.jpg"))
print(f"Extracted {len(image_names)} frames")
assert len(image_names) >= 2, "Need at least 2 frames — try a longer / clearer clip" 


In [ ]:
# 5) Load VGGT from Hugging Face and run reconstruction
import torch
from vggt.models.vggt import VGGT
from vggt.utils.load_fn import load_and_preprocess_images

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)
assert device == "cuda", "Enable GPU: Runtime → Change runtime type → T4 GPU, then Runtime → Run all"

model = VGGT.from_pretrained("facebook/VGGT-1B").to(device)
model.eval()

images = load_and_preprocess_images(image_names).to(device)
print("images tensor:", tuple(images.shape))

dtype = torch.bfloat16 if torch.cuda.get_device_capability()[0] >= 8 else torch.float16
with torch.no_grad():
    with torch.cuda.amp.autocast(dtype=dtype):
        predictions = model(images)

# Move tensors to CPU for export
for k, v in list(predictions.items()):
    if isinstance(v, torch.Tensor):
        predictions[k] = v.detach().cpu()

print("prediction keys:", sorted(predictions.keys()))


In [ ]:
# 6) Export GLB with VGGT's official helper
import sys
sys.path.insert(0, "/content/vggt_src")  # visual_util.py lives at repo root
from visual_util import predictions_to_glb

# visual_util expects images + world points style fields from the demo path.
# Ensure images are present for coloring.
scene = predictions_to_glb(
    predictions,
    conf_thres=50.0,
    filter_by_frames="all",
    show_cam=True,
    prediction_mode="Predicted Pointmap",
)
out_path = "/content/scene.glb"
scene.export(out_path)
print("Wrote", out_path)


In [ ]:
# 7) Download GLB → upload it in your Streamlit app for the same job
from google.colab import files
files.download("/content/scene.glb")
print("Done. In the app: Attach scene.glb to your job.")
